In [ ]:
# @title Prepare Environment
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

useQ6 = False # @param {"type":"boolean"}

if useQ6:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q6_K.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q6_K.gguf
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q5_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q5_0.gguf

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

# unet_loader = UNETLoader()
unet_loader = UnetLoaderGGUF()
# model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    """Save images as animated WEBP using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }

    with imageio.get_writer(
        output_path,
        format='WEBP',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    """Save images as WEBM using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }

    with imageio.get_writer(
        output_path,
        format='FFMPEG',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    """Save single frame as PNG image."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"

    frame = (image.cpu().numpy() * 255).astype(np.uint8)

    Image.fromarray(frame).save(output_path)

    return output_path

def generate_video(
    positive_prompt: str = "a fox moving quickly in a beautiful winter scenery nature trees mountains daytime tracking camera",
    negative_prompt: str = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走",
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    steps: int = 30,
    cfg_scale: float = 1.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 33,
    fps: int = 16,
    output_format: str = "mp4"
):

    with torch.inference_mode():
        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()

        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]

        print("Loading Unet Model...")
        if useQ6:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q6_K.gguf")[0]
        else:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q5_0.gguf")[0]
        # model = model_sampling.patch(model, 8)[0]

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]

            del vae
            torch.cuda.empty_cache()
            gc.collect()

            output_path = ""
            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], "ComfyUI")
                # print(f"Image saved as PNG: {output_path}")

                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(
                        decoded,
                        "ComfyUI",
                        fps=fps,
                        codec="vp9",
                        quality=10
                    )
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, "ComfyUI", fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")

                # print(f"Video saved as {output_format.upper()}: {output_path}")

                display_video(output_path)

        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    video_data = open(video_path,'rb').read()

    # Determine MIME type based on file extension
    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"  # default

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ Environment Setup Complete!")


In [ ]:
# --- Clean/Modern Gradio UI for 🔥 AI Porn Video Studio ---
import gradio as gr

# NSFW defaults and presets
NSFW_DEFAULT_POSITIVE = (
    "ultra high res, explicit uncensored, gorgeous pornstar, wet skin, neon photo studio, "
    "erotic pose, creamy perfect skin, luscious lips, photoreal, seductive, sparkly, moaning, "
    "stunning body, sweaty, deep focus, 8k, anime pop, camera flash, cumshot, realism"
)
NSFW_DEFAULT_NEGATIVE = (
    "censored, watermark, worst quality, deformed, blurry, jpeg artifacts, mosaic, cartoon, "
    "signature, artist name, bad composition, extra limbs, poorly rendered genitals"
)

NSFW_PRESETS = {
    "Solo Girl": "solo nude glamour, beautiful pornstar, bedroom, erotic pose, seductive eyes, soft lighting, tight body",
    "Doggy Style": "doggy style sex, nude couple, ass up, wet skin, full view, beautiful pornstar, high res, penetration",
    "Blowjob": "blowjob, hot girl sucking cock, facial, tongue out, glossy lips, closeup, intense expression",
    "Riding": "cowgirl, riding sex, bouncing tits, energetic, nude pornstar, moaning, POV",
    "Lesbian": "lesbian, beautiful naked girls, tongue kissing, groping, lingerie, intimacy, soft touch, wet skin",
    "Ahegao": "ahegao, tongue out, crazy eyes, drooling, orgasm face, hentai, exaggerated expression, cumshot, ultra high res",
    "Creampie": "creampie, cum dripping, deep penetration, perfect pussy, closeup, detailed, erotic"
}

# Define your CSS string here for end use
my_css = '''
body {background: #190018;}
.gradio-container {background:#180018;}
.big-header {color: #ff79c6; font-size:2.5em; font-weight:900; margin:0.5em 0 0.2em 0; text-align:center;}
.desc {color:#f8bee0;font-size:1.1em;text-align:center;margin-bottom:1.5em;}
.gr-textbox textarea {font-size:1.119em;}
.gr-button--primary {font-size:1.25em; padding:15px 40px; border-radius:22px; background:linear-gradient(90deg,#ff27b7,#7f33ff);border:none;box-shadow:0 0 24px #a03ccc99;}
.preset-row button {margin:0.15em 0.6em 0.3em 0;padding:10px 23px;border-radius:19px;border:1.15px solid #b138ff77;background:#3a153d;font-weight:700;color:#e1b8f9;font-size:1.03em;}
.preset-row button:active {background:#58218d;}
'''

def run_comfyui(
    pos_prompt,
    neg_prompt,
    frames,
    steps,
    width,
    height,
    use_q6,
    seed,
):
    import random
    # Note: Must set global useQ6 in main env cell for Q6 to take effect.
    global useQ6
    useQ6 = use_q6
    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**63-1)
    else:
        seed = int(seed)
    return generate_video(
        positive_prompt=pos_prompt,
        negative_prompt=neg_prompt,
        width=int(width),
        height=int(height),
        frames=int(frames),
        steps=int(steps),
        cfg_scale=1.0,
        sampler_name="uni_pc",
        scheduler="simple",
        fps=16,
        output_format="mp4",
        seed=int(seed)
    )

# ------- Gradio UI ----------
with gr.Blocks(css=my_css) as demo:  # CSS argument moved here for compatibility
    gr.Markdown("""
    <div class='big-header'>🔥 AI Porn Video Studio</div>
    <div class='desc'>Generate explicit NSFW short videos using free T4/Colab with Wan2.1-14b GGUF. <br>Choose a preset or enter your wildest fantasy!</div>
    """)
    
    with gr.Row():
        positive_box = gr.Textbox(
            label="Positive Prompt",
            value=NSFW_DEFAULT_POSITIVE,
            lines=7,
            max_lines=14,
            elem_classes=["gr-textbox"],
            show_copy_button=True,
            placeholder="Describe your wildest scene, e.g.: ultra high res, pornstar, neon lights, ..."
        )
    gr.Markdown("<b>Quick Presets</b>", elem_id=None, elem_classes=None)
    with gr.Row(elem_classes=["preset-row"]):
        preset_buttons = []
        for lbl, val in NSFW_PRESETS.items():
            btn = gr.Button(lbl)
            preset_buttons.append((btn, val))
    def preset_click(preset_text):
        return gr.update(value=preset_text)
    # --- FIXED LOOP ---
    for btn, val in preset_buttons:
        btn.click(
            fn=lambda x=val: preset_click(x), # Use lambda with a default arg to capture 'val'
            inputs=[],
            outputs=positive_box,
            queue=False
        )
    with gr.Row():
        negative_box = gr.Textbox(
            label="Negative Prompt",
            value=NSFW_DEFAULT_NEGATIVE,
            lines=4,
            max_lines=10,
            elem_classes=["gr-textbox"],
            show_copy_button=True,
            placeholder="What should the AI avoid? e.g.: watermark, ugly, censored, ..."
        )

    gr.Markdown("---")

    gr.Markdown("<b>Settings</b>")
    with gr.Row():
        frames = gr.Slider(8, 16, value=8, label="Frames (8~16)", step=1)
        steps = gr.Slider(12, 30, value=18, label="Sampling Steps (12~30)", step=1)
        width = gr.Slider(432, 960, value=480, label="Width (px)", step=16)
        height = gr.Slider(448, 1152, value=832, label="Height (px)", step=16)
        use_q6 = gr.Checkbox(label="Use Q6 Model (better for big videos, a bit slower)", value=False)
        seed = gr.Number(label="Seed (-1=random)", value=-1, precision=0)

    gr.Markdown("---")

    with gr.Row():
        gen_btn = gr.Button("💦 Generate", variant="primary")

    output_video = gr.Video(label="AI Porn Video Result", mirror_webcam=False, elem_id="outvid", interactive=False)

    # Run and display in place
    gen_btn.click(
        run_comfyui,
        inputs=[positive_box, negative_box, frames, steps, width, height, use_q6, seed],
        outputs=output_video,
        show_progress=True
    )

    gr.Markdown("""
    <div style='text-align:center;color:#faf0fa;margin-top:1em;background:#2a1435b3;padding:1em 1em;border-radius:24px;font-size:1em;'>
    Free GPU: T4, set Frames/Resolution lower for faster renders. For big video, use Q6 model and lower Res.<br>
    <b>NSFW only. For SFW, use another notebook.</b>
    </div>
    """)

# Move CSS into Blocks definition. No css argument in launch().
demo.queue().launch(share=True)
